# 911 Transcript Dataset Pipeline

Pipeline controller notebook. Orchestrates:
1. **Phase 0** — APD taxonomy analysis → supported category set for supervised labeling
2. **Phase 1** — Real 911 recordings → Whisper transcription → postprocess → label → dataset
3. **Phase 2** — APD distribution summary
4. **Phase 3** — Synthetic transcript generation to reach ~4000 total samples

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_EXTRACTION_ROOT = PROJECT_ROOT / "data_extraction"
for module_root in (PROJECT_ROOT, DATA_EXTRACTION_ROOT):
    if str(module_root) not in sys.path:
        sys.path.insert(0, str(module_root))

DATA_DIR = PROJECT_ROOT / "data"
APD_CSV = DATA_DIR / "apd_calls.csv"
RECORDINGS_DIR = DATA_DIR / "recordings"
METADATA_CSV = RECORDINGS_DIR / "metadata.csv"
FINAL_CATEGORIES_PATH = DATA_DIR / "final_categories.txt"
DISTRIBUTION_CSV = DATA_DIR / "category_severity_distribution.csv"
DATASET_CSV = DATA_DIR / "dispatch_dataset.csv"
TRANSCRIPT_DIR = DATA_DIR / "transcripts"
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

CEILING = 4000
MIN_PER_CATEGORY = 100
SEED = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"APD CSV exists: {APD_CSV.is_file()}")
print(f"Recordings dir exists: {RECORDINGS_DIR.is_dir()}")
print(f"Ceiling: {CEILING}, Min per category: {MIN_PER_CATEGORY}")


In [ ]:
# Module imports
from audio_transcript_generation import generate_whisper_transcript, postprocess_transcript
from apd_distribution_analysis import find_categories, find_distribution
from apd_distribution_analysis.sampling import compute_generation_plan, sample_category
from apd_distribution_analysis.analysis import (
    FINAL_CATEGORY_COLUMN,
    priority_to_severity,
    _normalize_text,
    _normalize_category,
    )
from synthetic_data_generation import generate_synthetic_transcript, label_transcript_with_context
from dataset_setup import add_record

import json
import pandas as pd
import csv

print("All modules imported successfully.")

In [ ]:
# Phase 0: derive supported categories from APD before any 911 labeling
categories = find_categories(APD_CSV, output_path=FINAL_CATEGORIES_PATH)
valid_categories = sorted(c for c in categories if c.lower() != "missing")

apd_categories_df = pd.read_csv(APD_CSV, usecols=[FINAL_CATEGORY_COLUMN])
missing_total = int(
    apd_categories_df[FINAL_CATEGORY_COLUMN]
    .map(_normalize_category)
    .eq("missing")
    .sum()
 )

dist = find_distribution(APD_CSV, output_path=DISTRIBUTION_CSV)
category_totals = dist["category_totals"]
severity_totals = dist["severity_totals"]
apd_df = dist["apd_df"]

supported_categories = sorted(
    cat
    for cat, count in category_totals.items()
    if cat != "Other" and count >= MIN_PER_CATEGORY
 )
unsupported_categories = {
    cat: count
    for cat, count in category_totals.items()
    if cat == "Other" or count < MIN_PER_CATEGORY
 }

print(f"Found {len(valid_categories)} APD categories")
print(f"DATA NOT CATEGORIZED: {missing_total:,}")
categorized_total = sum(category_totals[cat] for cat in supported_categories)
print(f"CATEGORIZED DATA: {categorized_total:,}")
for c in valid_categories:
    print(f"  - {c}")

print(f"\nSupported categories for labeling/training ({len(supported_categories)}):")
for c in supported_categories:
    print(f"  - {c}")

if unsupported_categories:
    print("\nExcluded from supervised labeling/generation policy:")
    for cat, count in sorted(unsupported_categories.items(), key=lambda x: x[1], reverse=True):
        reasons = []
        if cat == "Other":
            reasons.append("category removed from supervised taxonomy")
        if count < MIN_PER_CATEGORY:
            reasons.append(f"APD count {count} < {MIN_PER_CATEGORY}")
        print(f"  - {cat}: {', '.join(reasons)}")

print(f"\nSeverity totals:")
for sev in sorted(severity_totals.keys()):
    print(f"  Severity {sev}: {severity_totals[sev]:,}")

print(f"\nTotal APD rows: {len(apd_df):,}")

## Phase 1 — Real 911 Recordings

For each recording in `911_metadata.csv`:
1. Transcribe audio with Whisper (`generate_whisper_transcript`)
2. Add speaker labels (`postprocess_transcript`)
3. Predict dispatch labels using the APD-supported category set from Phase 0 (`label_transcript_with_context`)
4. Write record to dataset (`add_record` with `audio_recording_id`)

In [ ]:
metadata_df = pd.read_csv(METADATA_CSV)
rows_with_files = metadata_df.dropna(subset=["file_name"])
print(f"Total recordings in metadata: {len(rows_with_files)}")

# Check which already have .txt transcripts
already_transcribed = [
    row["file_name"]
    for _, row in rows_with_files.iterrows()
    if (TRANSCRIPT_DIR / f"{Path(str(row['file_name'])).stem}.txt").is_file()
]
print(f"Already transcribed: {len(already_transcribed)}")
print(f"Remaining: {len(rows_with_files) - len(already_transcribed)}")

In [ ]:
# Build set of already-recorded transcript names to skip labeling for completed rows
already_recorded: set[str] = set()
if DATASET_CSV.is_file() and DATASET_CSV.stat().st_size > 0:
    import csv as _csv
    with open(DATASET_CSV, "r", encoding="utf-8") as _f:
        for _row in _csv.DictReader(_f):
            if _row.get("transcript_file_name"):
                already_recorded.add(_row["transcript_file_name"])

In [ ]:
# Phase 1: Transcribe, postprocess, label, and record real recordings
completed = 0
errors = []
total = len(rows_with_files)

for idx, (_, row) in enumerate(rows_with_files.iterrows(), 1):
    fname = str(row["file_name"]).strip()
    audio_path = RECORDINGS_DIR / fname
    transcript_name = f"{Path(fname).stem}.txt"
    transcript_path = TRANSCRIPT_DIR / transcript_name
    recording_id = str(row.get("id", idx))

    # Step 1 & 2: Transcribe + postprocess (skip if .txt already exists)
    if transcript_path.is_file():
        # Skip entirely if already in dataset
        if transcript_name in already_recorded:
            completed += 1
            if idx % 25 == 0 or idx == total:
                print(f"  [{idx}/{total}] completed={completed} errors={len(errors)}")
            continue
        transcript_text = transcript_path.read_text(encoding="utf-8")
    elif audio_path.is_file():
        try:
            raw = generate_whisper_transcript(audio_path)
            transcript_text = postprocess_transcript(raw)
            transcript_path.write_text(transcript_text, encoding="utf-8")
        except Exception as exc:
            errors.append((fname, str(exc)))
            print(f"  [{idx}/{total}] ERROR transcribing {fname}: {exc}")
            continue
    else:
        errors.append((fname, "Audio file not found"))
        continue

    # Step 3: Label within the APD-supported category set
    try:
        extra_info = json.dumps({
            "title": str(row.get("title", "") or ""),
            "civilian_initiated": row.get("civilian_initiated"),
            "deaths": row.get("deaths"),
            "description": str(row.get("description", "") or ""),
        })
        labels = label_transcript_with_context(
            transcript_text,
            extra_info=extra_info,
            allowed_categories=supported_categories,
        )
    except Exception as exc:
        errors.append((fname, f"labeling failed: {exc}"))
        print(f"  [{idx}/{total}] ERROR labeling {fname}: {exc}")
        continue

    # Step 4: Add to dataset
    add_record(
        DATASET_CSV,
        transcript_file=transcript_name,
        severity=labels.get("severity", 1) if labels.get("severity") is not None else 1,
        category=labels["category"],
        dispatch_police=labels.get("dispatch_police", False),
        dispatch_emt=labels.get("dispatch_emt", False),
        dispatch_fire=labels.get("dispatch_fire", False),
        audio_recording_id=recording_id,
    )
    already_recorded.add(transcript_name)
    completed += 1

    if idx % 25 == 0 or idx == total:
        print(f"  [{idx}/{total}] completed={completed} errors={len(errors)}")

print(f"\nPhase 1 done. {completed} recorded, {len(errors)} errors.")
if errors:
    for fname, err in errors[:10]:
        print(f"  - {fname}: {err}")

## Phase 2 — Synthetic Dataset Expansion

Generate synthetic transcripts to reach the target dataset size (~4000).

Loop:
1. `get_sample()` — draw APD row maintaining distribution
2. `generate_synthetic_transcript()` — create fictional transcript
3. `label_transcript_with_context()` — predict missing labels
4. `add_record()` — write to dataset with `apd_id`

In [ ]:
# Recompute generation plan after Phase 1 so deficits reflect updated dataset counts.
raw_plan = compute_generation_plan(
    apd_df,
    DATASET_CSV,
    ceiling=CEILING,
    min_per_category=MIN_PER_CATEGORY,
 )

excluded_generation_categories = {
    cat: p
    for cat, p in raw_plan.items()
    if cat not in supported_categories
 }
plan = {
    cat: p
    for cat, p in raw_plan.items()
    if cat in supported_categories
 }

total_deficit = sum(p["deficit"] for p in plan.values())
total_existing = sum(p["existing"] for p in plan.values())
plan_apd_total = sum(p["apd_total"] for p in plan.values())

print(f"{'Category':<30} {'APD%':>5} {'Prop':>5} {'Target':>6} {'Have':>5} {'Avail':>6} {'Deficit':>7}")
print("-" * 75)
for cat in sorted(plan, key=lambda c: plan[c]["apd_total"], reverse=True):
    p = plan[cat]
    pct = p["apd_total"] / plan_apd_total * 100 if plan_apd_total else 0.0
    flag = ">>>" if p["deficit"] > 0 else ""
    print(f"  {cat:<28} {pct:>4.1f}% {p['proportional']:>5} {p['target']:>6} {p['existing']:>5} {p['available']:>6} {p['deficit']:>7} {flag}")

if excluded_generation_categories:
    print("\nExcluded from generation/training policy:")
    for cat in sorted(excluded_generation_categories, key=lambda c: excluded_generation_categories[c]["apd_total"], reverse=True):
        p = excluded_generation_categories[cat]
        reasons = []
        if cat == "Other":
            reasons.append("category removed from supervised taxonomy")
        if p["apd_total"] < MIN_PER_CATEGORY:
            reasons.append(f"APD count {p['apd_total']} < {MIN_PER_CATEGORY}")
        print(f"  - {cat}: {', '.join(reasons)}")

cats_needing = sum(1 for p in plan.values() if p["deficit"] > 0)
print(f"\n{cats_needing} supported categories need generation, {len(plan) - cats_needing} already at target.")

if total_deficit == 0:
    print("All supported categories meet their targets. No generation needed.")
else:
    print(f"\nTotal records to generate: {total_deficit}")
    print(f"Estimated final supported dataset size: ~{total_existing + total_deficit}")

In [ ]:
# Phase 3: APD-grounded synthetic generation (resume-safe, no duplicates)

if total_deficit <= 0:
    print("No generation needed.")
else:
    # Collect already-used APD IDs for dedup
    used_apd_ids: set[str] = set()
    if DATASET_CSV.is_file() and DATASET_CSV.stat().st_size > 0:
        with open(DATASET_CSV, "r", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                aid = row.get("apdId", "").strip()
                if aid:
                    used_apd_ids.add(aid)

    completed = 0
    errors = []

    for category in sorted(plan):
        deficit = plan[category]["deficit"]
        if deficit <= 0:
            continue

        # Safety guard: only generate within the supported taxonomy.
        if category not in supported_categories:
            continue

        # Sample unique APD rows for this category
        rows = sample_category(
            apd_df, category, deficit,
            exclude_ids=used_apd_ids,
            seed=SEED,
        )

        print(f"\n[{category}] generating {len(rows)} (target deficit={deficit})")

        for i, row in enumerate(rows, 1):
            apd_id = str(row.get("Incident Number", "")).strip()
            if not apd_id or apd_id in used_apd_ids:
                continue  # safety: skip if no ID or already used
            used_apd_ids.add(apd_id)

            severity = priority_to_severity(row.get("Priority Level"))

            try:
                transcript = generate_synthetic_transcript(
                    apd_id=apd_id,
                    incident_type=_normalize_text(row.get("Incident Type"), ""),
                    severity=severity,
                    mental_health_flag=_normalize_text(row.get("Mental Health Flag"), ""),
                    response_day=_normalize_text(row.get("Response Day of Week"), ""),
                    response_hour=str(row.get("Response Hour", "")),
                    category=category,
                    initial_problem_description=_normalize_text(row.get("Initial Problem Description", ""), ""),
                    sector=_normalize_text(row.get("Sector"), ""),
                    council_district=_normalize_text(row.get("Council District"), ""),
                    output_dir=TRANSCRIPT_DIR,
                )

                known_labels = {"severity": severity, "category": category}
                total_injury_death = (
                    int(row.get("Officer Injured/Killed Count") or 0)
                    + int(row.get("Subject Injured/Killed Count") or 0)
                    + int(row.get("Other Injured/Killed Count") or 0)
                )
                extra_info = json.dumps({
                    "Sector": str(row.get("Sector", "") or ""),
                    "Final Problem Description": str(row.get("Final Problem Description", "") or ""),
                    "Number of Units Arrived": row.get("Number of Units Arrived"),
                    "totalInjuryorDeath": total_injury_death,
                })

                labels = label_transcript_with_context(
                    transcript,
                    known_labels=known_labels,
                    extra_info=extra_info,
                    allowed_categories=supported_categories,
                )

                add_record(
                    DATASET_CSV,
                    transcript_file=f"{apd_id}.txt",
                    severity=labels["severity"] if labels.get("severity") is not None else severity,
                    category=labels["category"],
                    dispatch_police=labels["dispatch_police"],
                    dispatch_emt=labels["dispatch_emt"],
                    dispatch_fire=labels["dispatch_fire"],
                    apd_id=apd_id,
                )
                completed += 1

            except Exception as exc:
                errors.append((apd_id, str(exc)))
                print(f"    [{i}/{len(rows)}] ERROR {apd_id}: {exc}")

            if (completed + len(errors)) % 50 == 0:
                print(f"  progress: {completed + len(errors)}/{total_deficit} (completed={completed}, errors={len(errors)})")

    print(f"\nPhase 3 done. {completed} generated, {len(errors)} errors.")

## Phase 3 - Dataset Inspection

In [ ]:
if DATASET_CSV.is_file():
    df = pd.read_csv(DATASET_CSV)
    print(f"Total records: {len(df)}")
    print(f"\nSource breakdown:")
    real_count = df["audioRecordingId"].astype(bool).sum()
    synth_count = df["apdId"].astype(bool).sum()
    print(f"  Real recordings:    {real_count}")
    print(f"  Synthetic (APD):    {synth_count}")
    print(f"\nSeverity distribution:")
    print(df["severity"].value_counts().sort_index())
    print(f"\nCategory distribution (top 10):")
    print(df["category"].value_counts().head(10))
    print(f"\nDispatch flags:")
    for col in ["hasDispatchedPolice", "hasDispatchedEMT", "hasDispatchedFire"]:
        print(f"  {col}: {df[col].sum()} / {len(df)}")
    print(f"\nFirst 5 rows:")
    display(df.head())
else:
    print("Dataset CSV not found yet.")